# Vault Collection — geoid-only retrieval

**Use case**: a collection where features are persisted but unreachable without the geoid returned at write time. The geoid acts as a capability token. No filter search, no listing, no spatial query, no tiles, no STAC `/search` — losing the geoid means losing the feature.

## Routing shape

| Operation | Driver(s) | Purpose |
|---|---|---|
| **WRITE** | `items_postgresql_driver` (sync, fatal) + `items_elasticsearch_private_driver` (sync, warn) | PG = authoritative geometry; ES = geoid-keyed index for fast lookup |
| **READ** | `items_postgresql_driver` (`hints={"geometry_exact"}`) | Authoritative un-simplified geometry |
| **SEARCH** | `items_elasticsearch_private_driver` | Geoid lookup only — driver capabilities exclude `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` |

## Two retrieval modes

1. **Always-available — accurate geometry**: `GET /features/catalogs/{cat}/collections/{coll}/items/{geoid}` → READ driver = PG with `geometry_exact` hint → un-simplified. This is the OGC API Features standard endpoint; it's mounted whenever writes are. The writer must remember `(catalog_id, collection_id, geoid)`.
2. **Optional — fast cross-collection lookup**: `GET /search/catalogs/{cat}/geoid/{geoid}` → ES private index. Only available when the deployment includes the `search` extension (`dynastore[extension_search]`, part of `catalog_routes_grp`). Caller only needs `(catalog_id, geoid)` because the response carries `collection_id`. Returns `simplification_factor`/`simplification_mode` so the caller can decide whether to fall back to mode (1) for un-simplified geometry.

The notebook probes the `/search` extension at the start and degrades gracefully to mode (1) when it's absent. Mode (1) alone is sufficient for the vault contract — mode (2) is a convenience.

## Lockdown — isolated to a dedicated test principal (issue #209.3)

A bundle of regex-scoped DENY policies (registered via `POST /admin/policies?catalog_id={cat}`) blocks every discovery surface — features list, STAC search, tiles, EDR, coverages, connected_systems. Paired ALLOW policies open exactly the geoid-lookup, items-by-id, and write paths.

**Isolation contract**: the policy bundle is bound to a *dedicated test principal* (`vault-test-{cat}-evaluator`), never to the default `sysadmin / admin / user / anonymous` roles via hierarchy. Earlier revisions wired the vault role as a parent of every default role; a notebook crash before cleanup leaked the catalog-scoped DENYs into every principal's effective policy set globally. This revision:

- creates the test principal with `roles=[vault-{cat}]` only,
- never touches role hierarchies,
- in step 7 verifies that the **sysadmin** session (which never had the vault role) sees the discovery surfaces as `ALLOW` — proving the bundle has not bled into the default-role tree,
- guards the entire flow with a final cleanup cell that runs even if upstream cells errored (best-effort, idempotent).

Demonstrating that the lockdown actually fires for the test principal requires a token minted for that principal. Where the deployment exposes a local-user token endpoint, set `VAULT_TEST_TOKEN` (or rely on the `local_provider`'s `create_user` flow); otherwise step 7 reports `SKIP (enforcement)` and only the isolation half of the contract is verified inside the notebook.

## Pyodide notes

Runs in JupyterLite. The setup cell auto-fetches a sysadmin token from Keycloak when env-var tokens are absent (Pyodide has no shell environment). All HTTP via `httpx.AsyncClient`.

In [ ]:
import os
import json
import asyncio
import httpx

from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=False)
BASE = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080").rstrip("/")

# Auto-provision DYNASTORE_TOKEN via IDP client_credentials if not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass
KEYCLOAK = os.environ.get("KEYCLOAK_TOKEN_URL", "http://localhost:8180/realms/geoid/protocol/openid-connect/token")
CATALOG_ID = "demo-vault"
COLLECTION_ID = "vault-locations"

TOKEN = (
    os.environ.get("DYNASTORE_SYSADMIN_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or os.environ.get("DYNASTORE_TOKEN")
    or ""
)
if not TOKEN:
    print("No env token — fetching one from Keycloak (testadmin)…")
    async with httpx.AsyncClient(timeout=15.0) as kc:
        r = await kc.post(KEYCLOAK, data={
            "grant_type": "password",
            "client_id": "geoid-api",
            "client_secret": "geoid-api-secret",
            "username": "testadmin",
            "password": "testpassword",
        })
    if r.status_code == 200:
        TOKEN = r.json().get("access_token", "")
        print(f"Got token from Keycloak (len={len(TOKEN)})")
    else:
        print(f"Keycloak token request failed: {r.status_code} — {r.text[:200]}")

headers = {"Content-Type": "application/json"}
if TOKEN:
    headers["Authorization"] = f"Bearer {TOKEN}"
client = httpx.AsyncClient(base_url=BASE, headers=headers, timeout=120.0)


def _check(r, label=""):
    status = r.status_code
    ok = status < 400
    body = ""
    try:
        body = r.text[:300]
    except Exception:
        pass
    print(f"{label + ': ' if label else ''}{status}{'' if ok else f' — {body}'}")
    return r


# Probe whether the `search` extension is mounted on this deployment.
# `/search/catalogs/{any}/geoid/{any}` is the canonical fast-lookup route
# but it's only available when SCOPE includes `search`. Without it, the
# notebook uses the always-available items-by-id path instead.
try:
    _probe = await client.get("/search/catalogs/_probe_/geoid/00000000-0000-0000-0000-000000000000")
    SEARCH_AVAILABLE = _probe.status_code in (200, 400, 404)  # any answer means the route is mounted
except Exception:
    SEARCH_AVAILABLE = False

print(f"BASE             = {BASE}")
print(f"TOKEN            = {'set (' + str(len(TOKEN)) + ' chars)' if TOKEN else 'EMPTY (writes will fail)'}")
print(f"SEARCH ext       = {'mounted (fast-lookup available)' if SEARCH_AVAILABLE else 'absent (will use items-by-id only)'}")

## 1. Create catalog + vault collection (clean slate)

Vault collections live in their own catalog. Tile / EDR / connected_systems routes are **catalog-scoped**, not collection-scoped — keeping vault collections in a dedicated catalog lets the catalog-wide DENY policies (registered in step 3) cover those leak surfaces without affecting unrelated public collections.

In [ ]:
_cleanup = await client.delete(f"/stac/catalogs/{CATALOG_ID}", params={"force": "true"}, timeout=60.0)
if _cleanup.status_code not in (204, 404):
    print(f"WARN: cleanup returned {_cleanup.status_code}: {_cleanup.text[:200]}")

r = await client.post("/stac/catalogs", json={
    "id": CATALOG_ID,
    "title": "Vault Demo",
    "description": "Vault collections — geoid-only retrieval, no discovery.",
})
_check(r, "Catalog")

for _ in range(60):
    r = await client.get(f"/stac/catalogs/{CATALOG_ID}")
    if r.status_code == 200 and r.json().get("provisioning_status") == "ready":
        print("Catalog ready")
        break
    await asyncio.sleep(1)
else:
    print("WARN: catalog still provisioning after 60s")

r = await client.post(f"/stac/catalogs/{CATALOG_ID}/collections", json={
    "id": COLLECTION_ID,
    "title": "Vault Locations",
    "description": "Sensitive geometries reachable only by geoid.",
    "extent": {
        "spatial": {"bbox": [[-180, -90, 180, 90]]},
        "temporal": {"interval": [["2020-01-01T00:00:00Z", None]]},
    },
})
_check(r, "Collection")

## 2. PUT `collection_routing_config` — vault routing

- WRITE fan-out: PG (sync, fatal) + private ES (sync, warn). Loss of the authoritative store fails the ingest; loss of the indexer warns and fixes itself on next reindex.
- READ pinned to PG with the `geometry_exact` hint — accurate geometry on the items-by-id path.
- SEARCH pinned to private ES.

> **Important — routing waterfall**: collection-scoped routing config **merges** with catalog → platform → code defaults rather than replacing them. After the PUT, GET-back will show that SEARCH still includes other drivers (e.g., `items_postgresql_driver`, `items_duckdb_driver`) inherited from the platform default. The private ES driver's lack of `SPATIAL_FILTER`/`ATTRIBUTE_FILTER`/`FULLTEXT` capabilities means filter/spatial queries are still structurally broken at the driver layer, but **the route stays open**. **Real lockdown of SEARCH must come from IAM (step 3)** — the routing config is necessary but not sufficient by itself.

In [ ]:
routing = {
    "enabled": True,
    "operations": {
        "WRITE": [
            {"driver_ref": "items_postgresql_driver",
             "on_failure": "fatal", "write_mode": "sync"},
            {"driver_ref": "items_elasticsearch_private_driver",
             "on_failure": "warn",  "write_mode": "sync"},
        ],
        "READ": [
            {"driver_ref": "items_postgresql_driver",
             "hints": ["geometry_exact"]},
        ],
        "SEARCH": [
            {"driver_ref": "items_elasticsearch_private_driver"},
        ],
    },
}
r = await client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
    json=routing,
)
_check(r, "PUT collection_routing_config")

r = await client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/plugins/collection_routing_config",
)
ops = r.json().get("operations", {})
print("\nApplied routing.operations:")
for op, entries in ops.items():
    summary = [f"{e.get('driver_ref')}({','.join(e.get('hints') or []) or '-'})" for e in (entries or [])]
    print(f"  {op:8s} {summary}")

## 3. Apply the IAM lockdown bundle (policies + dedicated test principal)

Catalog admins register policies via `POST /admin/policies?catalog_id={cat}`. The `catalog_id` query param scopes the policy to that catalog (the API stores it as the policy's `partition_key`). DENY short-circuits ALLOW.

**Critical**: a registered policy is only *evaluated* when it is bound to a role the caller's principal holds. Registration alone is not enforcement. The pattern below:

1. Create the policy bundle (DENYs + ALLOWs).
2. Create a `vault-{cat}` Role that contains every policy id.
3. Create a **dedicated test principal** (`vault-test-{cat}-evaluator`) and assign it the vault role.

This deliberately replaces the previous parent-of-default-roles pattern (issue #209.3). Binding the vault role into the `sysadmin/admin/user/anonymous` hierarchy meant a notebook crash before cleanup left the DENYs leaking into every effective principal globally — and the catalog-wide tile/EDR/coverages DENYs would silently break unrelated work elsewhere on the deployment.

The actions list contains HTTP-method regexes (`GET`, `POST`, …); resources are URL-path regexes.

In [ ]:
CAT = CATALOG_ID
VAULT_ROLE = f'vault-{CAT}'
TEST_PRINCIPAL_SUBJECT = f'vault-test-{CAT}-evaluator'

policies = [
    # --- DENY discovery surfaces ---
    {'id': f'{CAT}-vault-deny-features-list', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [
         f'^/features/catalogs/{CAT}/collections/[^/]+/items/?$',
         f'^/features/catalogs/{CAT}/collections/[^/]+/queryables/?$',
     ]},
    {'id': f'{CAT}-vault-deny-stac-discovery', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/stac/catalogs/{CAT}/(search|collections-search)/?$',
         f'^/stac/catalogs/{CAT}/collections/[^/]+/items/?$',
     ]},
    {'id': f'{CAT}-vault-deny-records-list', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [f'^/records/catalogs/{CAT}/collections/[^/]+/items/?$']},
    {'id': f'{CAT}-vault-deny-tiles', 'effect': 'DENY', 'actions': ['GET'],
     'resources': [f'^/tiles/catalogs/{CAT}(/.*)?$']},
    {'id': f'{CAT}-vault-deny-edr-coverages-misc', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/edr/catalogs/{CAT}(/.*)?$',
         f'^/coverages/catalogs/{CAT}(/.*)?$',
         f'^/consys/catalogs/{CAT}(/.*)?$',
         f'^/wfs/catalogs/{CAT}(/.*)?$',
         f'^/maps/catalogs/{CAT}(/.*)?$',
     ]},
    {'id': f'{CAT}-vault-deny-search-fanout', 'effect': 'DENY', 'actions': ['GET', 'POST'],
     'resources': [
         f'^/search/catalogs/{CAT}/?$',
     ]},
    # --- ALLOW only the vault-permitted surfaces ---
    {'id': f'{CAT}-vault-allow-geoid-lookup', 'effect': 'ALLOW', 'actions': ['GET', 'POST'],
     'resources': [f'^/search/catalogs/{CAT}/geoid(/[^/]+)?/?$']},
    {'id': f'{CAT}-vault-allow-items-by-id', 'effect': 'ALLOW', 'actions': ['GET'],
     'resources': [f'^/features/catalogs/{CAT}/collections/[^/]+/items/[^/]+/?$']},
    {'id': f'{CAT}-vault-allow-writes', 'effect': 'ALLOW',
     'actions': ['POST', 'PUT', 'PATCH', 'DELETE'],
     'resources': [
         f'^/features/catalogs/{CAT}/collections/[^/]+/items(/[^/]+)?/?$',
         f'^/stac/catalogs/{CAT}/collections/[^/]+/items(/[^/]+)?/?$',
         f'^/configs/catalogs/{CAT}(/.*)?$',
         f'^/stac/catalogs/{CAT}(/collections(/[^/]+)?)?/?$',
     ]},
]

# Step 1 — register policies. The /admin/policies handler reads partition
# scope from the `?catalog_id=` query param, so the body no longer carries
# `partition_key`.
for p in policies:
    await client.delete(f"/admin/policies/{p['id']}", params={'catalog_id': CAT})
    r = await client.post('/admin/policies', params={'catalog_id': CAT}, json=p)
    print(f"  policy {p['effect']:5s} {p['id']:55s} -> {r.status_code}")

# Step 2 — create the vault role with all policy ids attached
policy_ids = [p['id'] for p in policies]
await client.delete(f'/admin/roles/{VAULT_ROLE}')
r = await client.post('/admin/roles', json={
    'name': VAULT_ROLE,
    'description': f'Vault lockdown bundle for catalog {CAT}',
    'policies': policy_ids,
})
print(f"  role create {VAULT_ROLE} -> {r.status_code}")

# Step 3 — create the dedicated test principal and bind the vault role to it.
# We deliberately do NOT register a role hierarchy. Earlier revisions of this
# notebook made `vault-{cat}` a parent of `sysadmin/admin/user/anonymous`,
# which propagated the DENY bundle into every principal's effective policy
# set globally. If the notebook errored before cleanup, the catalog-wide
# tile / EDR / coverages DENYs persisted and silently broke unrelated
# catalogs (issue #209.3).
#
# Principal-create shape: /admin/principals accepts raw principals when no
# password is supplied (provider != 'local' OR password absent). We pass
# `subject_id` explicitly so the synthetic identity is reproducible across
# reruns; `username` is required by the model and mirrored to subject_id.
TEST_PRINCIPAL_ID = None

_create_resp = await client.post('/admin/principals', json={
    'username': TEST_PRINCIPAL_SUBJECT,
    'subject_id': TEST_PRINCIPAL_SUBJECT,
    'provider': 'local',
    'roles': [VAULT_ROLE],
})
if _create_resp.status_code in (200, 201):
    TEST_PRINCIPAL_ID = _create_resp.json().get('id')
    print(f'  principal {TEST_PRINCIPAL_SUBJECT} -> {_create_resp.status_code} (id={TEST_PRINCIPAL_ID})')
elif _create_resp.status_code == 409:
    # Already exists from a previous run — look it up and update its roles.
    # /admin/principals takes ?q= (OGC API - Records §7.7 free-text partial match).
    print(f'  principal {TEST_PRINCIPAL_SUBJECT} already exists — updating roles')
    _list = await client.get(
        '/admin/principals', params={'q': TEST_PRINCIPAL_SUBJECT, 'provider': 'local'},
    )
    if _list.status_code == 200:
        for entry in _list.json() or []:
            if entry.get('subject_id') == TEST_PRINCIPAL_SUBJECT and entry.get('provider') == 'local':
                TEST_PRINCIPAL_ID = entry.get('id')
                break
    if TEST_PRINCIPAL_ID:
        _upd = await client.put(
            f'/admin/principals/{TEST_PRINCIPAL_ID}',
            json={'roles': [VAULT_ROLE]},
        )
        print(f'  principal {TEST_PRINCIPAL_ID} role update -> {_upd.status_code}')
else:
    print(f'  principal {TEST_PRINCIPAL_SUBJECT} -> {_create_resp.status_code} {_create_resp.text[:200]}')

if TEST_PRINCIPAL_ID:
    print(f"\nVault role '{VAULT_ROLE}' is bound to principal id={TEST_PRINCIPAL_ID} only.")
    print("Default roles (sysadmin/admin/user/anonymous) DO NOT inherit it.")
else:
    print("\nWARN: test principal could not be created; step 7 will skip the enforcement probe.")
print('Step 7 verifies (a) sysadmin still sees the discovery surfaces (isolation),')
print('and (b) — when VAULT_TEST_TOKEN is set — that the test principal sees them as 403.')

## 4. Write 2 features — small + large geometry

The large feature has hundreds of vertices. ES (the indexer) has a 10MB per-doc soft limit; the private driver simplifies geometries above the threshold via `simplify_to_fit` (`elasticsearch.py:460`). The PG store is unaffected. Step 5 reads from ES (may be simplified); step 6 reads from PG (always full).

In [ ]:
import math

# Small feature: 2 vertices. NOTE: no `id` field — let the platform assign a
# fresh UUID geoid. The capability-token model needs a system-assigned id;
# supplying our own would let an attacker guess the token by knowing our naming.
SMALL = {
    'type': 'Feature',
    'geometry': {'type': 'LineString', 'coordinates': [[0.0, 0.0], [0.1, 0.1]]},
    'properties': {'label': 'small', 'sensitivity': 'low'},
}

# Large feature: dense polygon, ~5_000 vertices around a circle. Forces ES
# simplification (private ES has a 10MB-per-doc soft limit; simplify_to_fit
# kicks in above the threshold). PG stores the un-simplified polygon.
N = 5000
ring = [
    [10.0 + 0.5 * math.cos(2 * math.pi * i / N), 20.0 + 0.5 * math.sin(2 * math.pi * i / N)]
    for i in range(N)
]
ring.append(ring[0])
LARGE = {
    'type': 'Feature',
    'geometry': {'type': 'Polygon', 'coordinates': [ring]},
    'properties': {'label': 'large', 'sensitivity': 'high', 'vertex_count': N},
}

fc = {'type': 'FeatureCollection', 'features': [SMALL, LARGE]}
r = await client.post(
    f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items',
    json=fc,
)
_check(r, 'Write 2 features')

body = r.json() if r.status_code < 400 else {}
ids = body.get('ids') or body.get('accepted_ids') or []
print('\nAssigned geoids (capability tokens — keep them safe!):')
for g in ids:
    print(f'  {g}')
GEOIDS = ids

## 5. Optional fast lookup via `/search/.../geoid/{geoid}` (ES path)

The geoid endpoint hits the private ES index. Returns full feature payload including `simplification_factor` and `simplification_mode`. For the small feature, `simplification_mode = "none"`. For the large feature, ES will have simplified the geometry to fit — `simplification_mode != "none"`.

This step requires the `search` extension. When it is absent the cell skips and step 6 retrieves both features from PG directly (the always-available path). The geoid is the only key either way.

In [ ]:
es_results = {}
if not SEARCH_AVAILABLE:
    print('Skipping — `search` extension is not mounted on this deployment.')
    print('Step 6 will retrieve both features from PG via items-by-id (always available).')
else:
    for g in GEOIDS:
        r = await client.get(f'/search/catalogs/{CATALOG_ID}/geoid/{g}')
        if r.status_code != 200:
            print(f'  {g}: {r.status_code} {r.text[:200]}')
            continue
        coll = r.json()
        res_list = coll.get('results') or []
        if not res_list:
            print(f'  {g}: empty result — the private ES index has no record for this geoid.')
            print(f'           Cause: items_elasticsearch_private_driver write fan-out did not produce an index entry')
            print(f'           on this stack (e.g., private indexer not enabled, index name mismatch).')
            print(f'           Step 6 (PG accurate retrieval) is unaffected — PG write always succeeds first.')
            continue
        res = res_list[0]
        geom = res.get('geometry') or {}
        coords = geom.get('coordinates') or []
        if geom.get('type') == 'Polygon' and coords:
            v = len(coords[0])
        elif geom.get('type') == 'LineString':
            v = len(coords)
        else:
            v = -1
        print(
            f'  {g}\n'
            f'    geometry.type   = {geom.get("type")}\n'
            f'    vertex_count    = {v}\n'
            f'    simplification  = mode={res.get("simplification_mode")!r} factor={res.get("simplification_factor")}\n'
            f'    collection_id   = {res.get("collection_id")!r}'
        )
        es_results[g] = res

## 6. Accurate geometry via the items-by-id round-trip (PG path, `geometry_exact` hint)

The ES result already carries `collection_id`, so we can chain a second call to `GET /features/catalogs/{cat}/collections/{coll}/items/{geoid}`. That handler resolves the READ driver via `get_driver(Operation.READ, ...)`; the routing config we PUT in step 2 baked `hints={"geometry_exact"}` into the PG entry, so the items-by-id call routes to PG and returns the un-simplified geometry.

In [ ]:
# If es_results is populated, use the collection_id from the ES record
# (the realistic flow). If it's empty (private indexer not active on this
# stack), fall back to the known COLLECTION_ID — the PG path still works
# because PG was the WRITE primary.
for g in GEOIDS:
    es_res = es_results.get(g) or {}
    coll_id = es_res.get('collection_id') or COLLECTION_ID
    r = await client.get(f'/features/catalogs/{CATALOG_ID}/collections/{coll_id}/items/{g}')
    if r.status_code != 200:
        print(f'  {g}: {r.status_code} {r.text[:200]}')
        continue
    feat = r.json()
    geom = feat.get('geometry') or {}
    coords = geom.get('coordinates') or []
    if geom.get('type') == 'Polygon' and coords:
        v_pg = len(coords[0])
    elif geom.get('type') == 'LineString':
        v_pg = len(coords)
    else:
        v_pg = -1
    es_geom = es_res.get('geometry') or {}
    es_coords = es_geom.get('coordinates') or []
    if es_geom.get('type') == 'Polygon' and es_coords:
        v_es = len(es_coords[0])
    elif es_geom.get('type') == 'LineString':
        v_es = len(es_coords)
    else:
        v_es = None
    if v_es is None:
        note = '  (no ES record on this stack — PG retrieval is the canonical path)'
    elif v_pg > v_es:
        note = f'  ← PG returned {v_pg - v_es} more vertices than ES (un-simplified)'
    else:
        note = '  (no simplification at ES — geometries match)'
    print(f'  {g}  ES={v_es if v_es is not None else "—":>5}  PG={v_pg:>5} vertices{note}')

## 7. Lockdown verification — isolation + (optional) enforcement

This step has two halves:

**Isolation (always runs)** — the sysadmin session was used for every preceding step. Sysadmin is NOT bound to the `vault-{cat}` role, so the discovery surfaces should still respond as `ALLOW` for sysadmin even after the bundle was registered. If any sysadmin probe returns `403`, the bundle has leaked into a default-role hierarchy and the isolation contract from issue #209.3 is broken.

**Enforcement (skipped without `VAULT_TEST_TOKEN`)** — when a token for the dedicated test principal is available, repeat the probes with that token and confirm the discovery surfaces return `403` while the geoid-lookup ALLOW surface still works. This half cannot be exercised in pure JupyterLite because the local provider has no in-notebook token-mint flow; set `VAULT_TEST_TOKEN` from a parent shell or fall back to operator verification.

A bogus geoid against the geoid-lookup ALLOW surface should return `404` regardless of which token is used (proves the endpoint is reachable; the geoid value itself is unknown).

In [ ]:
probes = [
    ('GET',  f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items',
     'features list', True),
    ('GET',  f'/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/queryables',
     'features queryables', True),
    ('POST', f'/stac/catalogs/{CATALOG_ID}/search', 'STAC search', True),
    ('POST', f'/stac/catalogs/{CATALOG_ID}/collections-search', 'STAC collections search', True),
    ('GET',  f'/tiles/catalogs/{CATALOG_ID}/0/0/0.mvt', 'tile leak', True),
    ('GET',  f'/edr/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/position?coords=POINT(0 0)',
     'EDR position', True),
    # Search-extension probes — only meaningful when the extension is mounted.
    ('GET',  f'/search/catalogs/{CATALOG_ID}', 'search items',
     SEARCH_AVAILABLE),
    ('GET',  f'/search/catalogs/{CATALOG_ID}/geoid/00000000-0000-0000-0000-000000000000',
     'bogus geoid', SEARCH_AVAILABLE),
]


def _classify(label, status, body_text):
    """Return (verdict_label, ok_for_isolation, ok_for_enforcement)."""
    if label == 'bogus geoid':
        empty = False
        if status == 200:
            try:
                empty = len((json.loads(body_text) or {}).get('results') or []) == 0
            except Exception:
                pass
        return (
            'EXPECTED' if status == 404 or empty else f'UNEXPECTED ({status})',
            status == 404 or empty,
            status == 404 or empty,
        )
    return (
        f'STATUS {status}',
        status != 403,    # isolation: sysadmin should NOT be denied
        status == 403,    # enforcement: vault principal SHOULD be denied
    )


# ── 7a. Isolation probe (sysadmin token, the existing `client`) ─────────────
print('=== 7a. ISOLATION (sysadmin — should see discovery surfaces as ALLOW) ===')
isolation_failures = []
for method, path, label, applicable in probes:
    if not applicable:
        print(f'  ---- {method:4s} {path[:78]:78s}  -> SKIP ({label}: search ext not mounted)')
        continue
    if method == 'GET':
        r = await client.get(path)
    else:
        r = await client.post(path, json={})
    verdict, isolation_ok, _ = _classify(label, r.status_code, r.text)
    flag = 'OK' if isolation_ok else '**LEAK**'
    print(f'  {method:4s} {path[:78]:78s}  -> {r.status_code:3d}  ({label}: {flag} — {verdict})')
    if not isolation_ok:
        isolation_failures.append((method, path, label, r.status_code))

print()
if isolation_failures:
    print(f'❌ Isolation contract violated — {len(isolation_failures)} surface(s) returned 403 to sysadmin.')
    print('   The vault role bundle has leaked into the default-role hierarchy.')
    print('   Inspect the test principal\\'s role bindings via GET /admin/principals/{id}.')
else:
    print('✅ Isolation contract intact — sysadmin retains access to all discovery surfaces.')
    print('   The vault DENY bundle is bound only to the dedicated test principal.')

# ── 7b. Enforcement probe (test-principal token, optional) ───────────────────
VAULT_TEST_TOKEN = os.environ.get('VAULT_TEST_TOKEN', '')
print()
print('=== 7b. ENFORCEMENT (test principal — should see discovery surfaces as 403) ===')
if not VAULT_TEST_TOKEN:
    print('  SKIP — VAULT_TEST_TOKEN not set. Set it to a Bearer token minted for')
    print(f'         provider=local subject_id={TEST_PRINCIPAL_SUBJECT!r} to verify enforcement.')
    print('         Operator verification: log in as the test principal and re-run the probes.')
else:
    test_headers = dict(headers, Authorization=f'Bearer {VAULT_TEST_TOKEN}')
    async with httpx.AsyncClient(base_url=BASE, headers=test_headers, timeout=120.0) as test_client:
        for method, path, label, applicable in probes:
            if not applicable:
                print(f'  ---- {method:4s} {path[:78]:78s}  -> SKIP ({label}: search ext not mounted)')
                continue
            if method == 'GET':
                r = await test_client.get(path)
            else:
                r = await test_client.post(path, json={})
            verdict, _, enforcement_ok = _classify(label, r.status_code, r.text)
            flag = 'BLOCKED' if enforcement_ok else 'OPEN'
            print(f'  {method:4s} {path[:78]:78s}  -> {r.status_code:3d}  ({label}: {flag} — {verdict})')

print()
print('Interpretation:')
print('  7a OK / **LEAK** — sysadmin probe; LEAK means the role bundle escaped its scope.')
print('  7b BLOCKED       — DENY policy fired for the test principal. Lockdown verified.')
print('  7b OPEN          — DENY policy registered but not evaluated for this token. Two causes:')
print('                      (a) cache not yet flushed — re-run after a few seconds, or')
print('                      (b) the token does not actually carry the `vault-{cat}` role —')
print('                          inspect via GET /iam/me with the test principal\\'s token.')
print('  EXPECTED          — bogus-geoid behavior matches contract.')
print('  SKIP              — route is part of an extension not mounted on this deployment.')

## 8. Cleanup

In [ ]:
# Best-effort cleanup so re-running the notebook is idempotent.
# Safety net required by issue #209.3: every step is guarded so a partial
# upstream failure still tears down every artifact this notebook owns.
# Running it twice is harmless. Running it without any upstream cell
# executed (e.g. straight after kernel restart) is also harmless — the
# `globals()` guards turn missing names into no-ops instead of NameError.

g = globals()


async def _safe(coro_factory, label):
    try:
        r = await coro_factory()
        ok = getattr(r, 'status_code', None)
        print(f'  cleanup {label:55s} -> {ok}')
    except Exception as e:
        print(f'  cleanup {label:55s} -> ERR: {e!r}')


_client = g.get('client')
if _client is None:
    print('No `client` in scope — setup cell never ran. Nothing to clean up.')
else:
    _cat = g.get('CATALOG_ID')
    _vault_role = g.get('VAULT_ROLE') or (f'vault-{_cat}' if _cat else None)
    _principal_id = g.get('TEST_PRINCIPAL_ID')
    _policies = g.get('policies') or []

    # 1. Delete the dedicated test principal (releases the role binding).
    if _principal_id:
        await _safe(
            lambda: _client.delete(f'/admin/principals/{_principal_id}'),
            f'principal {_principal_id}',
        )

    # 2. Drop the vault role.
    if _vault_role:
        await _safe(
            lambda v=_vault_role: _client.delete(f'/admin/roles/{v}'),
            f'role {_vault_role}',
        )

    # 3. Drop the policies.
    for p in _policies:
        await _safe(
            lambda pid=p['id'], cat=_cat: _client.delete(
                f'/admin/policies/{pid}', params={'catalog_id': cat},
            ),
            f"policy {p['id']}",
        )

    # 4. Drop the catalog (cascades to collections + items).
    if _cat:
        await _safe(
            lambda: _client.delete(
                f'/stac/catalogs/{_cat}', params={'force': 'true'}, timeout=60.0,
            ),
            f'catalog {_cat}',
        )

    try:
        await _client.aclose()
    except Exception:
        pass

print('Done.')

## Evaluation & recommendations

### Why IAM + routing rather than a driver flag or a new schema field

- **Driver layer is the wrong place**. A driver-side gate would have to be duplicated in PG, ES public, ES private, DuckDB, Iceberg, BigQuery — and would still not cover catalog-scoped routes (tiles, EDR, coverages) where no per-collection driver is consulted. Drivers are platform-agnostic by design (`feedback_full_driver_abstraction.md`); putting access policy there violates the SSOT.
- **A new schema field** (`discovery_mode: "vault"`) was rejected in favour of implicit detection from the routing shape. The combination `WRITE includes elasticsearch_private AND SEARCH is empty (or only elasticsearch_private)` uniquely identifies vault — no other valid combo matches. The detection logic can be promoted into `_apply_deny_policy` (`elasticsearch_private/driver.py:457`) as a follow-up so the bundle is auto-emitted on routing-config-apply.
- **IAM is the correct chokepoint**. Single enforcement point at `IamMiddleware.dispatch` (`extensions/iam/middleware.py:92`); regex DENY short-circuits ALLOW (`policies.py:424`); `partition_key` scopes the policy to the catalog so a catalog admin can self-serve.

### Why the role binds to a dedicated test principal, not a hierarchy edge (issue #209.3)

The earlier revision called `POST /iam/governance/hierarchies` to make `vault-{cat}` a *parent* of every default role. That made the bundle effective for everyone — sysadmin, admin, user, anonymous — and the catalog-wide DENYs (tiles, EDR, coverages) were enforced for unrelated catalogs too. A notebook crash before the cleanup cell ran left those DENYs persisted in the live IAM tables, polluting every subsequent session on the deployment.

The fix is structural: bind the role only to a dedicated principal (`vault-test-{cat}-evaluator`). The blast radius of a botched run is now scoped to that principal — even if cleanup never runs, no other principal sees the DENYs. The safety-net cleanup cell is best-effort and idempotent, scrubbing only the artifacts this notebook owns (principal, role, policies, catalog).

### Cross-extension impact matrix

| Extension | Discovery routes | Covered by | Notes |
|---|---|---|---|
| Features | `/items`, `/queryables` | DENY rule 1 | Collection-scoped; precise |
| STAC     | `/search`, `/collections-search`, `/items` | DENY rule 2 | Catalog and collection scoped |
| Records  | `/items` | DENY rule 3 | Collection-scoped |
| Tiles    | `/{z}/{x}/{y}.mvt` | DENY rule 4 | **Catalog-scoped** — see deep dive below |
| EDR      | `/position`, `/area`, `/cube`, `/locations` | DENY rule 5 | Catalog-scoped |
| Coverages | `/coverage`, `/domainset`, `/rangetype` | DENY rule 5 | |
| Connected Systems | `/systems`, `/datastreams`, `/observations` | DENY rule 5 | |
| WFS / Maps | `*` | DENY rule 5 | Catalog-scoped legacy surfaces |
| Search   | `/search/catalogs/{cat}` | DENY rule 6 | Geoid lookup is the ALLOW surface |
| Search   | `/geoid/{geoid}` | ALLOW rule 1 | The canonical retrieval path |
| Features | `/items/{geoid}` | ALLOW rule 2 | Accurate-geometry round-trip |

### Tile-route deep dive

Tiles are catalog-scoped (`tiles_service.py:148`) — no `{collection_id}` path segment. The simple per-collection DENY does not exist for tiles. **This notebook adopts the operational guideline of one vault collection per catalog**, so the catalog-wide tile DENY (rule 4) blocks 100% of leaks. Same applies to EDR, coverages, connected_systems, WFS, maps. Filed follow-up: handler-side filter that skips collections whose routing shape matches the implicit-vault rule, so vault and public collections can coexist in one catalog.

### Comparison with the sibling notebooks

| Notebook | WRITE | READ | SEARCH | Discovery | Use case |
|---|---|---|---|---|---|
| `private_es_only` | private ES | private ES | private ES | open by default | Smallest footprint; no PG fallback |
| `private_es_plus_pg` | PG + private ES | PG | private ES | open by default | Two-store durability + private indexer |
| `collection_vault_geoid_only` (this) | PG + private ES | PG (`geometry_exact`) | private ES | **locked down via IAM (per-principal)** | Capability-token semantics; geoid is the only key |

### Recommendations (filed as follow-up issues)

1. **`/search/catalogs/{cat}/geoid/{geoid}?source=geometry_exact`** — fold the two-call accurate-retrieval pattern into a single endpoint by routing the lookup through PG when the hint is requested. Today the notebook does steps 5 + 6 separately.
2. **Auto-emit the IAM bundle in `_apply_deny_policy`** — detect the implicit-vault routing shape on routing-config-apply and register the DENY/ALLOW bundle automatically, attaching it to a deployment-managed service principal rather than a default-role hierarchy. The current notebook bundles them manually.
3. **Per-handler vault filter for tiles / EDR / coverages / maps / WFS / connected_systems** — so vault and public collections can coexist in one catalog without falling back to the dedicated-catalog guideline.
4. **HATEOAS suppression in parent listings** — vault collection IDs still appear in `list_collections_in_catalog` etc. Not a leak by itself (every operation against the collection 403s) but a tightening opportunity.
5. **In-notebook test-principal token mint** — wire the local provider's password flow to `/auth/token` (or expose an admin-only impersonation endpoint) so the notebook can run the enforcement half of step 7 end-to-end without a parent shell.

### Threat-model notes

- The **geoid is a bearer token**. Treat it like a secret — never log, transmit only over secure channels, rotate by `DELETE`+re-`POST` (a new geoid is assigned on every write).
- The bogus-geoid endpoint returning 404 (vs 403) is an intentional information-leak boundary — it confirms the endpoint is open without disclosing whether the token is valid. Acceptable since a uniform-random UUID is exhaustively unguessable.
- If a deployment also wires a public ES indexer for this catalog, vault features will leak there. The routing config is the SSOT — this notebook deliberately omits public ES, and an audit job should check every vault catalog's routing config periodically.